This script performs a sensitivity analysis of the `ratio_new_vs_all` variable by calculating quantile-based thresholds and visualizing its distribution. It creates a histogram and a survival curve showing how many development-area polygons remain at each threshold.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
from pathlib import Path

# =============================================================================
# CONFIGURATION
# =============================================================================

INPUT_GPKG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\New_Housing_Development_Areas_buildup_ratio.gpkg"
OUTPUT_PNG = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\New_Housing_Development_Areas\Thresholds\sensitivity_ratio_new_vs_all.png"

COL    = "ratio_new_vs_all"
N_BINS = 40
DPI    = 150

# Quantiles used as thresholds — computed automatically from the data
QUANTILES = [0.25, 0.50, 0.75, 0.90, 0.95]

# =============================================================================
# MAIN
# =============================================================================

gdf  = gpd.read_file(INPUT_GPKG, columns=[COL])  # load only the required column
data = gdf[COL].dropna()
n    = len(data)
print(f"✓ {len(gdf):,} polygons loaded  |  {n:,} with valid {COL}")

Path(OUTPUT_PNG).parent.mkdir(parents=True, exist_ok=True)

# Compute thresholds from quantiles
THRESHOLDS = [round(float(data.quantile(q)), 4) for q in QUANTILES]
print("Quantile thresholds:")
for q, t in zip(QUANTILES, THRESHOLDS):
    print(f"  P{int(q*100):2d} = {t:.4f}")

# Colors for threshold lines
colors = ["#E69F00", "#56B4E9", "#009E73", "#CC79A7", "#D55E00"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Sensitivity Analysis: ratio_new_vs_all (Quantile Thresholds)",
             fontsize=14, fontweight="bold", y=1.01)

# -----------------------------------------------------------------------------
# LEFT: Histogram
# -----------------------------------------------------------------------------
ax = axes[0]
ax.hist(data, bins=N_BINS, color="#4C72B0", edgecolor="white", linewidth=0.5, alpha=0.85)

for val, col, q in zip(THRESHOLDS, colors, QUANTILES):
    n_above = (data >= val).sum()
    ax.axvline(val, color=col, linewidth=1.6, linestyle="--",
               label=f"P{int(q*100)} = {val:.3f}  ->  {n_above:,} ({n_above/n*100:.0f}%)")

ax.set_xlabel("ratio_new_vs_all", fontsize=11)
ax.set_ylabel("Number of Polygons", fontsize=11)
ax.set_title("Histogram", fontsize=11)
ax.legend(fontsize=8, title="Threshold", title_fontsize=8)
ax.grid(axis="y", linestyle="--", alpha=0.4)
ax.spines[["top", "right"]].set_visible(False)

# Stats box
p25, p50, p75 = data.quantile([0.25, 0.50, 0.75])
stats_text = (
    f"n = {n:,}\n"
    f"Min  = {data.min():.3f}\n"
    f"P25  = {p25:.3f}\n"
    f"P50  = {p50:.3f}\n"
    f"P75  = {p75:.3f}\n"
    f"Max  = {data.max():.3f}"
)
ax.text(0.02, 0.97, stats_text, transform=ax.transAxes,
        fontsize=8, verticalalignment="top",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9))

# -----------------------------------------------------------------------------
# RIGHT: Survival curve — share of polygons remaining per threshold
# -----------------------------------------------------------------------------
ax2 = axes[1]

x_vals = np.linspace(0, 1, 500)
y_vals = [(data >= x).sum() / n * 100 for x in x_vals]

ax2.plot(x_vals, y_vals, color="#4C72B0", linewidth=2)
ax2.fill_between(x_vals, y_vals, alpha=0.12, color="#4C72B0")

for val, col, q in zip(THRESHOLDS, colors, QUANTILES):
    pct = (data >= val).sum() / n * 100
    ax2.axvline(val, color=col, linewidth=1.6, linestyle="--")
    ax2.axhline(pct, color=col, linewidth=0.8, linestyle=":", alpha=0.6)
    ax2.annotate(f"P{int(q*100)} ({val:.3f})\n{pct:.0f}% remaining",
                 xy=(val, pct), xytext=(val + 0.01, pct + 1.5),
                 fontsize=7.5, color=col)

ax2.set_xlabel("Threshold ratio_new_vs_all >=", fontsize=11)
ax2.set_ylabel("Remaining Polygons (%)", fontsize=11)
ax2.set_title("Share of Remaining Polygons per Threshold", fontsize=11)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 105)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))
ax2.grid(linestyle="--", alpha=0.4)
ax2.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig(OUTPUT_PNG, dpi=DPI, bbox_inches="tight")
plt.close(fig)  # free memory
print(f"✓ Saved: {OUTPUT_PNG}")
plt.show()